目的
* ネコとイヌの画像データと **畳み込みニューラルネットワーク（CNN）** を用いて分類を行う。(所要時間約45分間)
---
Objective
* Classify images of cats and dogs using image data and a **Convolutional Neural Network (CNN)**. (Estimated time: about 45 minutes)

準備
* GPUを用いて処理を行うために、上部のメニューバーの「ランタイム」→「ランタイムのタイプを変更」からハードウェアアクセラレータをGPUにしてください。
---
Preparation
* To use GPU for processing, go to the top menu bar and select “Runtime” → “Change runtime type”, then set the hardware accelerator to GPU.

データセット
* ネコとイヌのカラー画像

（旧）Kaggleデータセット: https://www.kaggle.com/datasets/karakaggle/kaggle-cat-vs-dog-dataset?resource=download

（新）Kaggle Cats and Dogs Dataset: https://www.microsoft.com/en-us/download/details.aspx?id=54765

---
Dataset
* Color images of cats and dogs

(Old) Kaggle Dataset: https://www.kaggle.com/datasets/karakaggle/kaggle-cat-vs-dog-dataset?resource=download

(New) Kaggle Cats and Dogs Dataset: https://www.microsoft.com/en-us/download/details.aspx?id=54765

流れ

1. データの読み込みと前処理(ImageDataGenerator)
2. CNNモデルの構築(Sequential)
3. コンパイル(損失関数・最適化手法の指定)
4. 学習(60エポック)
5. 結果
---
Workflow
1. Data loading and preprocessing (ImageDataGenerator)
2. Building the CNN model (Sequential)
3. Compiling the model (specifying the loss function and optimizer)
4. Training (60 epochs)
5. Results

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Deep learning/3data

Mounted at /content/drive
/content/drive/MyDrive/Deep learning/3data


ImageDataGenerator
* リアルタイムにデータ拡張（水増し）しながらテンソル画像データのバッチを生成します。また、このジェネレータはデータを無限にループするので、無限にバッチを生成します。
* Generates batches of tensor image data while performing real-time data augmentation. This generator loops over the data indefinitely, so it continuously produces batches.

flow_from_directory
* generatorに対し、flow_from_directoryを使用して画像データを読み取らせます。
* ディレクトリへのパスを受け取り、拡張/正規化したデータのバッチを生成します。
---
* Use flow_from_directory with the generator to load image data.
* It takes a path to a directory and generates batches of augmented/normalized data.

In [ ]:
!pip install tensorflow

from tensorflow.keras.preprocessing.image import ImageDataGenerator

img_w, img_h = 150, 150        # 画像をリサイズするときのサイズ/Target size when resizing the image
batch_size = 32                # ミニバッチのサイズ/mini-batch size

# 訓練データを読み込んで加工するジェネレータを生成/Generate a generator that loads and preprocesses the training data
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,         # RGB値を0～1の範囲に変換/Convert RGB values to the range 0–1
    rotation_range=15,         # 15度の範囲でランダムに回転させる/Rotate randomly within a range of 15 degrees
    zoom_range=0.2,            # ランダムに拡大/Randomly enlarge
    horizontal_flip=True       # 水平方向にランダムに反転/Random horizontal flip
    )

# テストデータを読み込むジェネレータを生成/Create a generator to load test data
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

# 訓練データをミニバッチの数だけ生成/Generate training data equal to the number of mini-batches
train_generator = train_datagen.flow_from_directory(
    '/content/drive/MyDrive/Deep learning/3data/train',               # 訓練データのフォルダ/Training data folder
    target_size=(img_w, img_h), # 画像をリサイズ/Resize the image
    batch_size=batch_size,      # ミニバッチのサイズ/mini-batch size
    class_mode='binary')        # 二値分類なのでbinaryを指定/Since this is a binary classification task, specify "binary"

# テストデータをミニバッチの数だけ生成/Generate test data equal to the number of mini-batches
validation_generator = test_datagen.flow_from_directory(
    '/content/drive/MyDrive/Deep learning/3data/validation',          # テストデータのフォルダ/Test data folder
    target_size=(img_w, img_h), # 画像をリサイズ/Resize the image
    batch_size=batch_size,      # ミニバッチのサイズ/mini-batch size
    class_mode='binary')        # 二値分類なのでbinaryを指定/Since this is a binary classification task, specify "binary"

Found 2000 images belonging to 2 classes.
Found 800 images belonging to 2 classes.


画像のラベルと番号を確認する方法
* ラベルは、flow_from_directoryで指定したディレクトリにある、**フォルダ**の名前が割り当てられます。
* ディレクトリに「dog」と「cat」という**フォルダ**があれば、ラベルは[dog, cat]になります。
* そのラベルには番号が割り振られています。
* それを確認するのがclass_indicesの機能です。
* label:番号で保存されています。
---
How to Check Image Labels and Their Indices
* Labels are assigned based on the names of the **folders** in the directory specified in flow_from_directory.
* If there are **folders** named “dog” and “cat” in the directory, the labels will be [dog, cat].
* Each label is assigned a numerical index.
* The class_indices attribute is used to check this mapping.
* Labels are stored in the format: label:number.

In [ ]:
# 正解ラベルを確認/Check the ground truth labels
print(train_generator.class_indices)
print(validation_generator.class_indices)

{'cats': 0, 'dogs': 1}
{'cats': 0, 'dogs': 1}


Sequentialモデル
* Kerasでモデルを作成するにはSequentialモデルを用いる方法とFunctionalAPIを用いる方法があります。
* Sequentialモデルでは、はじめにSequentialクラスのインスタンスを作成します。
* そしてaddメソッドを用いてレイヤ(全結合層やCNN層、プーリング層など)を追加していきます。
---
Sequential Model
* In Keras, there are two main ways to build models: using the Sequential model or the Functional API.
* With the Sequential model, you first create an instance of the Sequential class.
* Then, you use the add method to stack layers (such as Dense layers, CNN layers, pooling layers, etc.).

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense
from keras import optimizers

# モデルを構築/Seqentialモデルのインスタンスを作成/Build the model/Create an instance of the Sequential model
model = Sequential()

# （第1層）畳み込み層/addメソッドでレイヤを追加/(First layer) Convolutional layer / Add the layer using the add method
model.add(
    Conv2D(
        filters=32,                # フィルターの数は32/The number of filters is 32
        kernel_size=(3, 3),        # 3×3のフィルターを使用/Use a 3×3 filter
        input_shape=(150, 150, 3), # 入力データの形状/Shape of the input data
        padding='same',            # ゼロパディングを行う/Perform zero padding
        activation='relu'          # 活性化関数はReLU/The activation function is ReLU
        ))
# （第2層）プーリング層/(Second Layer) Pooling Layer
model.add(
    MaxPooling2D(pool_size=(2, 2))
)
# ドロップアウト25％/Dropout rate: 25%
model.add(Dropout(0.25))

# （第3層）畳み込み層/(Layer 3) Convolutional Layer
model.add(
    Conv2D(
        filters=32,            # フィルターの数は32/The number of filters is 32
        kernel_size=(3, 3),    # 3×3のフィルターを使用/Use a 3×3 filter
        activation='relu'      # 活性化関数はReLU/The activation function is ReLU
        ))
# （第4層）プーリング層/(Layer 4) Pooling Layer
model.add(
    MaxPooling2D(pool_size=(2, 2))
)
# ドロップアウト25％/Dropout rate: 25%
model.add(Dropout(0.25))

# （第5層）畳み込み層/(Layer 5) Convolutional Layer
model.add(
    Conv2D(filters=64,         # フィルターの数は64/The number of filters is 64
           kernel_size=(3, 3), # 3×3のフィルターを使用/Use a 3×3 filter
           activation='relu'   # 活性化関数はReLU/The activation function is ReLU
          ))
# （第6層）プーリング層/(Layer 6) Pooling Layer
model.add(
    MaxPooling2D(pool_size=(2, 2)))
# ドロップアウト25％/Dropout rate: 25%
model.add(Dropout(0.25))

# 出力層への入力を4階テンソルから2階テンソルに変換する/Reshape the input to the output layer from a 4D tensor to a 2D tensor
model.add(Flatten())

# （第7層）全結合層/(Layer 7) Fully Connected Layer
model.add(
    Dense(64,                  # ニューロン数は64/The number of neurons is 64
          activation='relu'))  # 活性化関数はReLU/The activation function is ReLU
# ドロップアウト50％/Dropout rate: 50%
model.add(Dropout(0.5))

# （第8層）出力層/(Layer 8) Output Layer
model.add(
    Dense(
        1,                     # ニューロン数は1個/The number of neurons is one
        activation='sigmoid')) # 活性化関数はsigmoid/The activation function is sigmoid

# モデルのコンパイル/Compile the model
model.compile(
    loss='binary_crossentropy', # バイナリ用の交差エントロピー誤差/Binary Cross-Entropy Loss
    metrics=['accuracy'],       # 学習評価として正解率を指定/Specify accuracy as the evaluation metric
    # Adamアルゴリズムで最適化/Optimize using the Adam algorithm
    optimizer=optimizers.Adam(),
)

# モデルのサマリを表示/Display the model summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 150, 150, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 75, 75, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 75, 75, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 73, 73, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 36, 36, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 17, 17, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 18496)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     1,183,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,212,513 (4.63 MB)

 Trainable params: 1,212,513 (4.63 MB)

 Non-trainable params: 0 (0.00 B)

ImageDataGeneratorを使った学習
* ImageDataGeneratorを利用して学習を行うには、model.fit()を使います。
---
Training with ImageDataGenerator
* To train a model using ImageDataGenerator, use model.fit().

In [ ]:
epochs = 60             # 学習回数/number of training epochs
batch_size = batch_size # 設定済みのミニバッチのサイズ/Configured mini-batch size

# 学習を行う/Train the model
history = model.fit(
    train_generator,    # 訓練データ/training data
    epochs=epochs,      # 学習回数/number of training epochs
    verbose=1,          # 学習の進捗状況を出力する/Output learning progress
    # テストデータ/test data
    validation_data=validation_generator,
)

Epoch 1/60
63/63 ━━━━━━━━━━━━━━━━━━━━ 706s 11s/step - accuracy: 0.5100 - loss: 0.7064 - val_accuracy: 0.5000 - val_loss: 0.6874
Epoch 2/60
63/63 ━━━━━━━━━━━━━━━━━━━━ 23s 359ms/step - accuracy: 0.5350 - loss: 0.6801 - val_accuracy: 0.5750 - val_loss: 0.6647
Epoch 3/60
63/63 ━━━━━━━━━━━━━━━━━━━━ 24s 382ms/step - accuracy: 0.5415 - loss: 0.6755 - val_accuracy: 0.5825 - val_loss: 0.6910
Epoch 4/60
63/63 ━━━━━━━━━━━━━━━━━━━━ 23s 374ms/step - accuracy: 0.5720 - loss: 0.6798 - val_accuracy: 0.5875 - val_loss: 0.6640
Epoch 5/60
63/63 ━━━━━━━━━━━━━━━━━━━━ 24s 374ms/step - accuracy: 0.6025 - loss: 0.6614 - val_accuracy: 0.5938 - val_loss: 0.6509
Epoch 6/60
63/63 ━━━━━━━━━━━━━━━━━━━━ 23s 372ms/step - accuracy: 0.6030 - loss: 0.6538 - val_accuracy: 0.6513 - val_loss: 0.6586
Epoch 7/60
63/63 ━━━━━━━━━━━━━━━━━━━━ 22s 353ms/step - accuracy: 0.6375 - loss: 0.6325 - val_accuracy: 0.6600 - val_loss: 0.6121
Epoch 8/60
63/63 ━━━━━━━━━━━━━━━━━━━━ 23s 367ms/step - accuracy: 0.6705 - loss: 0.5976 - val_accur

ネコとイヌの識別の正解率は85%。データ拡張しても精度は低い。

The classification accuracy for distinguishing cats and dogs is 85%. Even with data augmentation, the accuracy remains relatively low.